In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.bh.rdm.yzwlab.com/' #'https://admin.staging.rdm.example.com/'
rdm_url = 'https://bh.rdm.yzwlab.com/'

idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None

idp_name_quota_test_1 = None
idp_username_quota_test_1 = None
idp_password_quota_test_1 = None

# idp_username_quota_test_1 が所属する機関名（テスト中に機関ストレージをAmazon S3へ切り替え、終了時にNII Storageへ戻す）
target_organization = None
institution_default_max_quota = 5
s3_access_key = None
s3_secret_key = None
s3_bucket = None

# idp_username_quota_test_1 でログインした際にIdPから返されるメールアドレス（Adminのユーザ検索に使用）
email_search = None
# 破壊的操作(GDPR delete)を含む子notebookに引き継ぐ、本番相当URLパターン（必須）。
# 詳細は各子notebook(テスト手順-機関ストレージのクォータ-再ログイン_*)を参照。
production_url_forbidden_patterns = None

default_result_path = None
close_on_fail = False
transition_timeout = 60000
skip_failed_test = True
exclude_notebooks = []

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt='IdP name for integrated_admin')
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin}')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')
(len(idp_username_integrated_admin), len(idp_password_integrated_admin))

In [ ]:
if s3_access_key is None:
    s3_access_key = input(prompt='S3 Access Key')
if s3_secret_key is None:
    s3_secret_key = getpass(prompt='S3 Secret Key')
if s3_bucket is None:
    s3_bucket = input(prompt='S3 Bucket Name')
(len(s3_access_key), len(s3_secret_key), len(s3_bucket))

In [ ]:
if idp_name_quota_test_1 is None:
    idp_name_quota_test_1 = input(prompt='IdP name for quota_test_1')
if idp_username_quota_test_1 is None:
    idp_username_quota_test_1 = input(prompt=f'Username for {idp_name_quota_test_1}')
if idp_password_quota_test_1 is None:
    idp_password_quota_test_1 = getpass(prompt=f'Password for {idp_username_quota_test_1}@{idp_name_quota_test_1}')
if email_search is None:
    email_search = input(prompt='Email address of quota test user 1 (Admin ユーザ検索欄の入力値)')
if target_organization is None:
    target_organization = input(prompt='Target organization name for quota test (対象機関名)')
if production_url_forbidden_patterns is None:
    production_url_forbidden_patterns = input(prompt='本番相当環境と判定するURLパターン (カンマ区切り。GDPR delete等の破壊的操作の事前チェックに使用)')
if isinstance(production_url_forbidden_patterns, str):
    production_url_forbidden_patterns = [p.strip() for p in production_url_forbidden_patterns.split(',') if p.strip()]
assert production_url_forbidden_patterns, (
    'production_url_forbidden_patterns が未設定です。GDPR delete等の破壊的操作を含むテストのため、'
    '本番相当のURLパターンを1つ以上指定してください。'
)
(len(idp_username_quota_test_1), len(idp_password_quota_test_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# GakuNinRDM 総合テスト [機関ストレージのクォータ]

- サブシステム名: 機関ストレージのクォータ
- ページ/アドオン: 機関ストレージのクォータ
- 機能分類: 機関ストレージのクォータ
- シナリオ名: *
- 用意するテストデータ: アカウント(クォータテストユーザー1), アカウント(統合管理者)

In [ ]:
import os
from scripts.papermillHelpers import gen_run_notebook

def make_result_dir(base_path):
    result_dir = os.path.join(base_path, 'notebooks')
    os.makedirs(result_dir, exist_ok=True)
    return result_dir

result_dir = make_result_dir(default_result_path)

run_notebook = gen_run_notebook(
    result_dir,
    transition_timeout,
    dict(
        admin_rdm_url=admin_rdm_url,
        idp_name_integrated_admin=idp_name_integrated_admin,
        idp_username_integrated_admin=idp_username_integrated_admin,
        idp_password_integrated_admin=idp_password_integrated_admin,
        email_search=email_search,
        production_url_forbidden_patterns=production_url_forbidden_patterns,
        transition_timeout=transition_timeout,
        close_on_fail=close_on_fail,
    ),
    skip_failed_test,
    exclude_notebooks,
)

result_notebooks = []
result_dir

## 「再ログイン_NII」テストの実施

テスト「テスト手順-機関ストレージのクォータ-再ログイン_NII」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-機関ストレージのクォータ-再ログイン_NII.ipynb',
    dict(
        rdm_url=rdm_url,
        idp_name_quota_test_1=idp_name_quota_test_1,
        idp_username_quota_test_1=idp_username_quota_test_1,
        idp_password_quota_test_1=idp_password_quota_test_1,
        target_organization=target_organization,
    )
))
result_notebooks[-1]

## 「再ログイン_Institutions」テストの実施

テスト「テスト手順-機関ストレージのクォータ-再ログイン_Institutions」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    'テスト手順-機関ストレージのクォータ-再ログイン_Institutions.ipynb',
    dict(
        rdm_url=rdm_url,
        idp_name_quota_test_1=idp_name_quota_test_1,
        idp_username_quota_test_1=idp_username_quota_test_1,
        idp_password_quota_test_1=idp_password_quota_test_1,
        target_organization=target_organization,
        institution_default_max_quota=institution_default_max_quota,
        s3_access_key=s3_access_key,
        s3_secret_key=s3_secret_key,
        s3_bucket=s3_bucket,
    )
))
result_notebooks[-1]